# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all fields and entities by their Croissant `@id`. This enables robust, schema-driven data exploration in FAIR/linked-data style.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Get record set @ids
record_sets = dataset.record_set_ids
print(f"Record sets (@id): {record_sets}\n")
for record_set_id in record_sets:
    recset = dataset.find_by_id(record_set_id)
    print(f"Record Set @id: {record_set_id}")
    print(f"  Name: {getattr(recset, 'name', '<no name>')}")
    print(f"  Description: {getattr(recset, 'description', '<no description>')}")
    field_ids = getattr(recset, 'field_ids', [])
    print(f"  Fields: {field_ids}")
    # Print field descriptions
    for field_id in field_ids:
        field = dataset.find_by_id(field_id)
        print(f"    Field @id: {field_id}")
        print(f"      Name: {getattr(field, 'name', '<no name>')}")
        print(f"      Description: {getattr(field, 'description', '<no description>')}")
        print(f"      Data type: {getattr(field, 'data_type', '<no type>')}")
    print('-'*40)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, there is usually only one main record set, but let's generalize:
dataframes = dict()

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print('No records extracted for this record set.')
    print('-'*35)
# Choose a record set for further analysis:
if len(dataframes):
    main_record_set = list(dataframes.keys())[0]
    df = dataframes[main_record_set]
else:
    main_record_set = None
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All fields are referenced by their schema `@id`.


In [ ]:
if df is not None:
    print('Columns available:')
    print(df.columns.tolist())
    # Try to automatically select a numeric field for demonstration
    sample_numeric_field = None
    for col in df.columns:
        # Try to cast first 10 records
        try:
            sample_vals = pd.to_numeric(df[col].dropna()[:10])
            if len(sample_vals)>0 and sample_vals.dtype.kind in 'fi':
                sample_numeric_field = col
                break
        except:
            continue
    if not sample_numeric_field:
        print("No numeric field identified for EDA.")
    else:
        print(f"Using numeric field '@id': {sample_numeric_field}\n")

        # Drop non-numeric values
        numeric_data = pd.to_numeric(df[sample_numeric_field], errors='coerce')

        threshold = numeric_data.mean() if numeric_data.notnull().sum()>0 else 0

        filtered_df = df[numeric_data > threshold].copy()
        print(f"Filtered records where {sample_numeric_field} > {threshold:.3f}: {len(filtered_df)} records.")
        print(filtered_df.head())

        filtered_df[f"{sample_numeric_field}_normalized"] = (numeric_data - numeric_data.mean()) / numeric_data.std()

        print(f"\nFirst five normalized values for {sample_numeric_field}:")
        print(filtered_df[[sample_numeric_field, f"{sample_numeric_field}_normalized"]].head())

        # Try to group by another field (categorical)
        group_field = None
        for col in df.columns:
            if col != sample_numeric_field and df[col].nunique()<20:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by field '@id': {group_field}\n")
            grouped_df = filtered_df.groupby(group_field)[sample_numeric_field].mean()
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
else:
    print('No data available for EDA steps.')

## 5. Visualization
Visualize the distribution of the chosen numeric field, and, if present, how it differs across groups. Visualization is based on field `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if df is not None and sample_numeric_field is not None:
    plt.figure(figsize=(8, 5))
    numeric_data = pd.to_numeric(df[sample_numeric_field], errors='coerce')
    sns.histplot(numeric_data.dropna(), kde=True)
    plt.title(f"Histogram of field '@id': {sample_numeric_field}")
    plt.xlabel(sample_numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If a group_field is found, show boxplot
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=sample_numeric_field, data=df)
        plt.title(f"Distribution of {sample_numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading the FAIR² Mass Spectrometry dataset using `mlcroissant`, listing record sets and fields by their Croissant `@id`, and performing basic EDA and visualization. This approach ensures that all data operations are traceable to explicit schema definitions, promoting reproducible and FAIR research workflows.

_Next steps could include: advanced field selection by semantic type (using the Croissant schema), deeper groupwise statistics, or export to other interoperable formats._